# RFDiffusion3 Tutorial

This notebook walks through two examples of using the **RFDiffusion3 ProtFlow runner**.

| # | Example | Key concept |
|---|---------|-------------|
| 1 | De novo design | Running RFD3 without any input structure |
| 2 | Motif-conditioned design | Fixed residues, motif tracking |



## Imports and shared setup

We import ProtFlow, initialise the jobstarter, and create the runner once. Both examples reuse these objects.
Make sure you execute commands in an active ProtFlow environment.

In [ ]:
import logging
import os
import time

import pandas as pd

import protflow
from protflow.jobstarters import SbatchArrayJobstarter
from protflow.poses import Poses
from protflow.tools.rfdiffusion3 import RFdiffusion3
from protflow.residues import residue_selection

# logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)

# SbatchArrayJobstarter runs jobs by submitting a SLURM job
gpu_jobstarter = SbatchArrayJobstarter(max_cores=10, gpus=1, options="--time=00:10:00 ")

# single runner instance reused across all examples
rfdiffusion3 = RFdiffusion3(jobstarter=gpu_jobstarter)

# paths (relative to notebook location)
INPUT_PDB = 'rfdiffusion3_data/input/5AN7.pdb'
OUT_ROOT  = 'rfdiffusion3_data/outputs/'
os.makedirs(OUT_ROOT, exist_ok=True)

print('Setup complete.')
print(f'  Input PDB  : {os.path.abspath(INPUT_PDB)}')
print(f'  Output root: {os.path.abspath(OUT_ROOT)}')

---
## Example 1 — De novo design

In **de novo** mode we provide no input structure. RFD3 generates proteins from scratch guided only by a length range.

### How it works

- We create an **empty `Poses` object** by passing `poses=None`.
- The runner detects `len(poses) == 0` and enters de novo mode.
- It builds a minimal input JSON containing only the `length` field.
- `settings_group_name` defaults automatically to `"denovo"` — no need to set it.
- After the run, `poses.df` is populated directly from the collected output scores.

### Input JSON written internally by the runner
```json
{
    "denovo": {
        "length": "100-100"
    }
}
```

In [ ]:
OUT_DENOVO = os.path.join(OUT_ROOT, 'rfd3_denovo')
os.makedirs(OUT_DENOVO, exist_ok=True)

# poses=None signals that there is no input structure
poses_denovo = Poses(
    poses=None,
    work_dir=OUT_DENOVO,
    jobstarter=gpu_jobstarter,
)

print(f'Empty poses created: {len(poses_denovo.df.index)} rows (expected 0)')

In [ ]:
# length='100' means exactly 100 residues
# 1 batch x diffusion_batch_size=8  ->  8 output structures
# overwrite=True ensures a clean run even if outputs already exist
poses_denovo = rfdiffusion3.run(
    poses=poses_denovo,
    prefix='rfdiffusion3',
    length='100',
    n_batches=1,
    diffusion_batch_size=8,
    overwrite=True,
)

print(f'De novo run complete: {len(poses_denovo.df.index)} output poses')

In [ ]:
# The runner populates poses.df with one row per output structure.
# Key columns:
#   description   unique name derived from the output filename
#   location      absolute path to the decompressed .cif file
#   metrics_*     scores from the RFD3 sidecar JSON

print('Columns in poses.df:')
print(list(poses_denovo.df.columns))

poses_denovo.df[['description', 'location']]

In [ ]:
scores_path = os.path.join(OUT_DENOVO, 'scores_denovo.json')
poses_denovo.save_scores(scores_path)
print(f'Scores saved to: {scores_path}')

---
## Example 2 — Motif-conditioned design with 5an7.pdb

Here we use an **input structure** and **fix specific residues** so that RFD3 designs a new backbone around them while keeping those residues in place.

### Fixed residues

We fix four residues in chain A: `A1051`, `A1083`, `A1110`, `A1180`. All heavy atoms are constrained (`"ALL"`).

### Motif tracking

After diffusion the output backbone is renumbered from scratch, so our fixed residues will have different numbers in the output. The runner uses the `diffused_index_map` in the sidecar JSON to record exactly where each fixed residue ended up.

We track this by adding two columns to `poses.df` **before** the run:

| Column | Role |
|--------|------|
| `residues_original` | Input positions — **never overwritten** |
| `residues_postdiffusion` | Output positions — **overwritten by `remap_motifs` after each run** |

Passing `update_motifs=['residues_postdiffusion']` to `run()` tells the runner to fill that column with the new positions after diffusion.

### Input JSON written internally by the runner
```json
{
    "rfd3": {
        "input": "5AN7.pdb",
        "unindex": "A1051,A1083,A1110,A1180",
        "length": "180-220",
        "select_fixed_atoms": {
            "A1051": "ALL", "A1083": "ALL",
            "A1110": "ALL", "A1180": "ALL"
        }
    }
}
```

In [ ]:
OUT_MOTIF = os.path.join(OUT_ROOT, 'rfd3_motif_conditioned')
os.makedirs(OUT_MOTIF, exist_ok=True)

# load input poses: globs *.pdb from the directory (just 5an7.pdb here)
poses_motif = Poses(
    poses=os.path.dirname(INPUT_PDB),
    glob_suffix='*.pdb',
    work_dir=OUT_MOTIF,
    jobstarter=gpu_jobstarter,
)

print(f'Loaded {len(poses_motif.df.index)} input pose(s): {poses_motif.poses_list()}')

In [ ]:
FIXED_RESIDUES = 'A1051,A1083,A1110,A1180'
motif = residue_selection(FIXED_RESIDUES, delim=',')

print(f'Tracking residues: {motif.residues}')

# residues_original      -> never touched after this cell
# residues_postdiffusion -> overwritten by remap_motifs inside run()
poses_motif.df['residues_original']      = [motif for _ in poses_motif.poses_list()]
poses_motif.df['residues_postdiffusion'] = [motif for _ in poses_motif.poses_list()]

print('Motif tracking columns added to poses.df.')

In [ ]:
poses_motif = rfdiffusion3.run(
    poses=poses_motif,
    prefix='rfdiffusion3',
    settings_group_name='rfd3',
    ligand="LLK",
    unindex=FIXED_RESIDUES,
    length='180-220',
    select_fixed_atoms={
        'A1051': 'ALL',
        'A1083': 'ALL',
        'A1110': 'ALL',
        'A1180': 'ALL',
    },
    n_batches=1,
    diffusion_batch_size=1,
    update_motifs=['residues_postdiffusion'],
    overwrite=True,
)

print(f'Motif run complete: {len(poses_motif.df.index)} output poses')

In [ ]:
# residues_original  stays as A1051, A1083, A1110, A1180 for every row.
# residues_postdiffusion shows where those residues landed in each output.

print('Motif positions before and after diffusion:\n')
for i, row in poses_motif.df.iterrows():
    orig     = row['residues_original'].residues
    postdiff = row['residues_postdiffusion'].residues
    desc     = row.get('poses_description', str(i))
    print(f'  {desc}')
    print(f'    original      : {orig}')
    print(f'    postdiffusion : {postdiff}')
    print()

In [ ]:
# show all columns added by the runner
runner_cols = [c for c in poses_motif.df.columns if c.startswith('rfdiffusion3_')]
print(f'Columns added by runner ({len(runner_cols)} total):')
for c in runner_cols:
    print(f'  {c}')

In [ ]:
scores_path = os.path.join(OUT_MOTIF, 'scores_motif.json')
poses_motif.save_scores(scores_path)
print(f'Scores saved to: {scores_path}')

---
## Summary

| Example | `poses=` | `overwrite=` | Key parameter | Output |
|---------|----------|-------------|---------------|--------|
| 1 — De novo | `None` | `True` | `length='100-100'` | 8 structures, no input reference |
| 2 — Motif | PDB dir | `True` | `select_fixed_atoms`, `update_motifs` | 8 structures, remapped residue positions |

### Next steps

- Increase `diffusion_batch_size` or `n_batches` to generate more designs.
- Use `multiplex_poses=N` to saturate multiple GPUs from a single input.
- Pipe `poses` into a ProteinMPNN or ESMFold runner for a full design loop.
- Use `fail_on_missing_output_poses=True` in production to catch silent RFD3 crashes.